# Synthetische GPTNERMED-Daten mit Qwen (Colab GPU)

Erzeugt **5.000** synthetische deutsche Medizin-NER-Sätze (Labels: `Medikation`, `Dosis`, `Diagnose`) mit **Qwen2.5-3B-Instruct** auf einer Colab-GPU und speichert sie in **Google Drive** — im selben Format wie `jfrei/GPTNERMED` bzw. dein `generate_new_dataset.py`.

Die Sampling-/Prompt-/Span-Logik **spiegelt** `meaningful_modification/scripts/generate_new_dataset.py` (dort via Gemini). Unterschiede hier: lokales **Qwen** statt Gemini-API + **Batching** fürs Tempo.

> **Vor dem Start:** `Runtime → Change runtime type →` **T4 GPU** auswählen.

In [ ]:
# 1) GPU pruefen (muss eine GPU zeigen, sonst Runtime-Type auf T4 GPU stellen)
!nvidia-smi

## 2) Medizinische Listen bereitstellen

Die Generierung braucht **zwei CSVs** aus deinem Repo (`meaningful_modification/data/medical_information/`):

- `who_atc_ddd.csv` — Medikamente + Dosiseinheiten/-werte
- `ICD-10-GM.csv` — Diagnosen

Kopiere diese beiden Dateien **einmalig** nach Google Drive:
`MyDrive/gptnermed_synthetic/medical_information/`

*(Die Dateien liegen lokal bei dir, sind aber via `.gitignore` nicht im Repo — daher der Umweg über Drive. `Alpha-ID-SE.csv` wird hier nicht benötigt.)*

In [ ]:
# 3) Abhaengigkeiten (torch ist auf Colab vorinstalliert)
!pip install -q -U transformers accelerate datasets pandas

In [ ]:
# 4) Google Drive einbinden
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 5) Konfiguration
from pathlib import Path

MODEL_NAME     = "Qwen/Qwen2.5-3B-Instruct"
NUM_SENTENCES  = 5000     # Anzahl gueltiger Saetze
BATCH_SIZE     = 16       # bei genug VRAM erhoehbar (z.B. 32) -> schneller
MAX_NEW_TOKENS = 256
NUM_EXAMPLES   = 2        # Few-Shot-Beispiele pro Prompt
TEMPERATURE    = 0.7
TOP_P          = 0.8
SEED           = 42

DRIVE_ROOT = Path("/content/drive/MyDrive/gptnermed_synthetic")
MED_DIR    = DRIVE_ROOT / "medical_information"
OUT_DIR    = DRIVE_ROOT
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert (MED_DIR / "who_atc_ddd.csv").exists(), f"who_atc_ddd.csv fehlt in {MED_DIR}"
assert (MED_DIR / "ICD-10-GM.csv").exists(),  f"ICD-10-GM.csv fehlt in {MED_DIR}"
print("Listen gefunden in:", MED_DIR)

In [ ]:
# 6) Sampling / Prompt / Spans  (Spiegel von generate_new_dataset.py)
import json, re
import random
import pandas as pd
from datasets import ClassLabel, Dataset, Features, Sequence, Value

# Struktur (Reihenfolge/Anzahl) der Entitaeten pro Satz
SHAPES = [
    ("Medikation", "Dosis"),
    ("Medikation", "Dosis", "Diagnose"),
    ("Diagnose",),
    ("Medikation", "Diagnose"),
    ("Medikation", "Dosis", "Diagnose", "Diagnose"),
]

# 12 Original-Saetze aus dem GPTNERMED-Prompt (Few-Shot-Pool)
ORIGINAL_SENTENCES = [
    '<s>Zur weiteren Bekämpfung des <class="Diagnose">Juckreiz</class> wird die Einnahme von täglich <class="Dosis">100mg</class> <class="Medikation">Cortison</class> empfohlen.</s>',
    '<s>Bei wiederkehrender Infektion wie einer <class="Diagnose">Sepsis</class> oder schweren <class="Diagnose">Pnseumonien</class> wird eine Überwachung erforderlich sein.</s>',
    '<s><class="Medikation">Valsartan</class>/<class="Medikation">HCT</class> <class="Dosis">160</class>/<class="Dosis">12,5 mg</class> 1-0-0</s>',
    '<s><class="Medikation">Pantoprazol</class> <class="Dosis">40 mg</class> p.o.</s>',
    '<s>Die feingewebliche histopathologische Untersuchung ergab den Befund einer <class="Diagnose">Metastase</class> des bekannten malignen <class="Diagnose">Melanoms</class>.</s>',
    '<s><class="Diagnose">Diabetes Typ 2</class>-Patienten müssen regelmäßig <class="Medikation">Insulin</class> (mindestens mit <class="Dosis">12ml</class> dosiert) spritzen.</s>',
    '<s>Ich nehme <class="Medikation">Antibiotika</class> seit Tagen. Seitdem ist die <class="Diagnose">Mandelentzündung</class> deutlich besser geworden.</s>',
    '<s>Entlassung: <class="Dosis">40mg</class> <class="Medikation">Lidocain</class> wegen <class="Diagnose">Kopfschmerzen</class></s>',
    '<s>Zusammenfassende D: Zervix-PE bei 11 und 2 Uhr mit ausgeprägter <class="Diagnose">chronisch-florider Zervizitis</class>.</s>',
    '<s>Die Verschreibung von <class="Medikation">Hämatokrin</class> <class="Dosis">43mg</class> war unnötig.</s>',
    '<s>Der Patient klagt über <class="Diagnose">Karditiden</class> und nimmt täglich <class="Medikation">Nifedipin</class> ein.</s>',
    '<s>D: PE-Material der Portio bei 1 Uhr mit Nachweis einer schwergradigen <class="Diagnose">squamösen intraepithelialen Läsion</class> (<class="Diagnose">HSIL</class>; hier noch <class="Diagnose">CIN II</class>).</s>',
]

SYSTEM_PROMT = (
    "Du erzeugst genau einen synthetischen deutschen medizinischen Satz für das Training eines NER-Modells.\n"
    "Verbindliche Regeln:\n"
    "1. Verwende jeden unter „Begriffe“ angegebenen Ausdruck exakt in der vorgegebenen Schreibweise, einschließlich Groß-/Kleinschreibung, Satzzeichen und Leerzeichen.\n"
    "2. Jeder vorgegebene Ausdruck muss im Feld „text“ genau einmal vorkommen.\n"
    "3. Verwende ausschließlich die vorgegebenen medizinischen Informationen.\n"
    "4. Ergänze keine weiteren Medikamente, Wirkstoffe, Dosierungen, Diagnosen, Symptome, Indikationen, Applikationswege, Darreichungsformen, Zeitangaben oder Behandlungseigenschaften.\n"
    "5. Allgemeine nichtmedizinische Funktionswörter und neutrale Satzbestandteile sind erlaubt.\n"
    "6. Erzeuge genau einen grammatisch vollständigen, plausiblen deutschen Satz.\n"
    "7. Erzeuge für jeden vorgegebenen Begriff genau einen Eintrag in „entities“.\n"
    "8. Der Wert von „entities.text“ muss exakt mit dem entsprechenden Ausdruck im Satz übereinstimmen.\n"
    "9. Die Einträge in „entities“ müssen in derselben Reihenfolge stehen, in der die Begriffe im Satz vorkommen.\n"
    "10. Verwende ausschließlich diese Labels: „Medikation“, „Dosis“ und „Diagnose“.\n"
    "11. Verwende keine Entität, die nicht in den Eingabebegriffen enthalten ist.\n"
    "12. Prüfe vor der Ausgabe intern: Sind alle Begriffe exakt einmal enthalten? Wurden keine medizinischen Informationen ergänzt? Stimmen Text und Labels exakt überein? Ist das JSON syntaktisch gültig?\n"
    "13. Antworte ausschließlich mit einem einzelnen gültigen JSON-Objekt. Kein Markdown, keine Erklärung und kein zusätzlicher Text.\n"
)

DATASET_FEATURES = Features({
    "sentence": Value("string"),
    "ner_labels": Sequence(feature={
        "ner_class": ClassLabel(names=["Medikation", "Dosis", "Diagnose"]),
        "start": Value("int32"),
        "stop": Value("int32"),
    }),
})

def load_medical_data(med_dir):
    who = pd.read_csv(med_dir / "who_atc_ddd.csv").dropna(subset=["ddd", "uom"])
    icd = pd.read_csv(med_dir / "ICD-10-GM.csv", sep=";", encoding="utf-8-sig", dtype=str)
    MEDS       = who["atc_name"].astype(str).to_list()
    DOSE_UNITS = who["uom"].astype(str).to_list()
    DOSE_VAL   = who["ddd"].astype(str).to_list()
    DIAGS      = icd["Diagnose"].astype(str).to_list()
    return MEDS, DIAGS, DOSE_VAL, DOSE_UNITS

def make_dosage(rng, values, units):
    val, unit = rng.choice(values), rng.choice(units)
    text = f"{val}{unit}" if rng.random() < 0.7 else f"{val} {unit}"
    return text.replace(".", ",")

def sample_entities(rng, medication, diagnoses, dose_values, dose_units):
    entities = []
    for word in rng.choice(SHAPES):
        if word == "Medikation":
            entities.append({"text": rng.choice(medication), "label": "Medikation"})
        elif word == "Dosis":
            entities.append({"text": make_dosage(rng, dose_values, dose_units), "label": "Dosis"})
        elif word == "Diagnose":
            entities.append({"text": rng.choice(diagnoses), "label": "Diagnose"})
    return entities

CLASS_RE = re.compile(r'<class="([^"]+)">(.*?)</class>')
def parse_original_sentence(tagged):
    inner = tagged.replace("<s>", "").replace("</s>", "")
    entities = [{"text": m.group(2), "label": m.group(1)} for m in CLASS_RE.finditer(inner)]
    return {"text": CLASS_RE.sub(r"\2", inner), "entities": entities}

EXAMPLE_POOL = [parse_original_sentence(s) for s in ORIGINAL_SENTENCES]

def build_examples(rng, pool, num_examples=2):
    chosen = rng.sample(pool, num_examples)
    return "\n".join(["Beispiel Sätze:"] + [json.dumps(e, ensure_ascii=False) for e in chosen])

def build_task(entities):
    return "\n".join(["Begriffe:"] + [f"- {e['label']}: {e['text']}" for e in entities])

def build_format():
    return 'Format:\n{"text": "...", "entities": [{"text": "...", "label": "Medication|Dosis|Diagnose"}, ...]}'

def build_llm_prompt(rng, entities, num_examples=2):
    return "\n\n".join([build_task(entities), build_format(), build_examples(rng, EXAMPLE_POOL, num_examples)])

def get_label_id(label):
    return {"Medikation": 0, "Dosis": 1, "Diagnose": 2}.get(label)

def find_spans(text, entities):
    """case-insensitiv, fortlaufender Cursor (korrekt bei wiederholten Entitaeten)."""
    lower = text.lower()
    spans, cursor = [], 0
    for ent in entities:
        needle = ent["text"].lower()
        start = lower.find(needle, cursor)
        if start == -1:
            start = lower.find(needle)   # Fallback: Reihenfolge weicht ab
        stop = start + len(needle) if start != -1 else -1
        if start != -1:
            cursor = stop
        spans.append({"start": start, "stop": stop, "label": ent["label"]})
    spans.sort(key=lambda s: s["start"])
    return {
        "ner_class": [get_label_id(s["label"]) for s in spans],
        "start":     [s["start"] for s in spans],
        "stop":      [s["stop"]  for s in spans],
    }

def build_dataset_form(text, spans):
    return {"sentence": text, "ner_labels": spans}

print("Logik geladen.")

In [ ]:
# 7) Qwen laden (float16 fuer T4)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # decoder-only Batching braucht left-padding

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Modell geladen auf:", model.device)

In [ ]:
# 8) Gebatchte Generierung
@torch.no_grad()
def generate_batch(prompts, max_new_tokens=MAX_NEW_TOKENS):
    texts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMT},
             {"role": "user",   "content": p}],
            tokenize=False, add_generation_prompt=True,
        )
        for p in prompts
    ]
    inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(gen, skip_special_tokens=True)

In [ ]:
# 9) Hauptschleife: 5000 gueltige Saetze erzeugen (mit Oversampling)
import time

MEDS, DIAGS, DOSE_VAL, DOSE_UNITS = load_medical_data(MED_DIR)
rng = random.Random(SEED)

records, attempts = [], 0
MAX_ATTEMPTS = NUM_SENTENCES * 4
t0 = time.time()

while len(records) < NUM_SENTENCES and attempts < MAX_ATTEMPTS:
    batch_entities = [sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
                      for _ in range(BATCH_SIZE)]
    prompts = [build_llm_prompt(rng, ents, NUM_EXAMPLES) for ents in batch_entities]
    attempts += BATCH_SIZE

    try:
        outputs = generate_batch(prompts)
    except Exception as e:
        print("[batch-fail]", e)
        continue

    for ents, raw in zip(batch_entities, outputs):
        if len(records) >= NUM_SENTENCES:
            break
        raw = re.sub(r"^```\w*\s*|\s*```$", "", raw.strip())
        try:
            result = json.loads(raw)
            spans = find_spans(result["text"], ents)
        except Exception:
            continue
        if -1 in spans["start"]:
            continue
        records.append(build_dataset_form(result["text"], spans))

    elapsed = time.time() - t0
    rate = len(records) / elapsed if elapsed else 0.0
    eta_min = (NUM_SENTENCES - len(records)) / rate / 60 if rate else float("inf")
    print(f"{len(records)}/{NUM_SENTENCES} gültig | {attempts} Versuche "
          f"| {rate:.2f} Sätze/s | ETA ~{eta_min:.1f} min")

print(f"\nFertig: {len(records)} gültige Sätze in {(time.time()-t0)/60:.1f} min "
      f"({attempts} Versuche, {100*len(records)/max(attempts,1):.1f}% Trefferquote)")

In [ ]:
# 10) In Google Drive speichern (HF-Dataset + JSONL, GPTNERMED-Format)
dataset    = Dataset.from_list(records, features=DATASET_FEATURES)
ds_path    = OUT_DIR / "synthetic_gptnermed_qwen"
jsonl_path = OUT_DIR / "synthetic_gptnermed_qwen.jsonl"

dataset.save_to_disk(str(ds_path))
with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Gespeichert in Drive:")
print(" -", ds_path)
print(" -", jsonl_path)

## Lokal weiterverwenden

1. Aus Drive laden: `MyDrive/gptnermed_synthetic/synthetic_gptnermed_qwen` (bzw. `.jsonl`).
2. Ins Repo legen unter `meaningful_modification/data/synthetic/`.
3. Mit `gptnermed_data_preparation.py` nach `.spacy` konvertieren und in dein Training mischen — für die **Ablation** (mit/ohne Synthetik, per-Typ-F1).

**Hinweis Datenqualität:** `find_spans` matcht case-insensitiv über die (fortlaufende) Fundstelle. Bei ungewöhnlichen Oberflächenformen kann die Span-Grenze leicht danebenliegen — für maximal saubere Labels ggf. exaktes, offset-genaues Matching ergänzen.